In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

def trainer(cx, cy, cz, radius, height, a, b, d, show_links):
    t = np.linspace(0, 2*np.pi, 500)
    zz = np.linspace(cz, cz + height, 70)
    T, ZZ = np.meshgrid(t, zz)
    XX = cx + radius*np.cos(T)
    YY = cy + radius*np.sin(T)

    # Перетин площини з бічною поверхнею циліндра
    x_all = cx + radius*np.cos(t)
    y_all = cy + radius*np.sin(t)
    z_all = cz + a*radius*np.cos(t) + b*radius*np.sin(t) + d
    valid = (z_all >= cz) & (z_all <= cz + height)
    xs, ys, zs = x_all[valid], y_all[valid], z_all[valid]

    fig = plt.figure(figsize=(16, 9))
    axf = fig.add_subplot(221)
    axp = fig.add_subplot(222)
    axh = fig.add_subplot(223)
    ax3 = fig.add_subplot(224, projection="3d")

    # Фронтальна проекція XZ
    axf.set_title("ФРОНТАЛЬНА ПРОЕКЦІЯ XZ", fontweight="bold")
    axf.set_xlabel("X"); axf.set_ylabel("Z")
    axf.grid(alpha=.25); axf.set_aspect("equal", adjustable="box")
    axf.plot([cx-radius, cx+radius, cx+radius, cx-radius, cx-radius],
             [cz, cz, cz+height, cz+height, cz], color="black")
    if len(xs):
        axf.plot(xs, zs, color="red", linewidth=2)
        axf.scatter(xs, zs, color="red", s=8)

    # Профільна проекція YZ: графічно розвертаємо Y
    axp.set_title("ПРОФІЛЬНА ПРОЕКЦІЯ YZ", fontweight="bold")
    axp.set_xlabel("Y"); axp.set_ylabel("Z")
    axp.grid(alpha=.25); axp.set_aspect("equal", adjustable="box")
    axp.plot([cy-radius, cy+radius, cy+radius, cy-radius, cy-radius],
             [cz, cz, cz+height, cz+height, cz], color="black")
    if len(ys):
        axp.plot(ys, zs, color="red", linewidth=2)
        axp.scatter(ys, zs, color="red", s=8)
    axp.invert_xaxis()

    # Горизонтальна проекція XY
    axh.set_title("ГОРИЗОНТАЛЬНА ПРОЕКЦІЯ XY", fontweight="bold")
    axh.set_xlabel("X"); axh.set_ylabel("Y")
    axh.grid(alpha=.25); axh.set_aspect("equal", adjustable="box")
    circle_x = cx + radius*np.cos(t)
    circle_y = cy + radius*np.sin(t)
    axh.plot(circle_x, circle_y, color="black")
    if len(xs):
        axh.plot(xs, ys, color="red", linewidth=2)
        axh.scatter(xs, ys, color="red", s=8)

    if show_links and len(xs):
        indices = np.linspace(0, len(xs)-1, min(10, len(xs)), dtype=int)
        for i in indices:
            axf.plot([xs[i], xs[i]], [cz, zs[i]], "--", alpha=.25, linewidth=.8)
            axp.plot([ys[i], ys[i]], [cz, zs[i]], "--", alpha=.25, linewidth=.8)
            axh.plot([xs[i], xs[i]], [ys[i]-0.15, ys[i]+0.15], ":", alpha=.3)

    # 3D-модель циліндра
    ax3.set_title("3D: ЦИЛІНДР, СІЧНА ПЛОЩИНА І ПЕРЕРІЗ", fontweight="bold")
    ax3.plot_surface(XX, YY, ZZ, alpha=.14, linewidth=0, color="steelblue")

    # Площина, обрізана круглим перерізом і висотою циліндра
    grid = np.linspace(-radius, radius, 80)
    U, V = np.meshgrid(grid, grid)
    inside = U**2 + V**2 <= radius**2
    plane_z = cz + a*U + b*V + d
    plane_z = np.where(inside & (plane_z >= cz) & (plane_z <= cz+height), plane_z, np.nan)
    ax3.plot_surface(cx+U, cy+V, plane_z, alpha=.3, linewidth=0, color="orange")

    if len(xs):
        ax3.plot(xs, ys, zs, color="red", linewidth=3, label="Лінія перерізу")
        ax3.scatter(xs, ys, zs, color="red", s=8)

    ax3.plot(circle_x, circle_y, np.full_like(t, cz), color="black", linewidth=1)
    ax3.plot(circle_x, circle_y, np.full_like(t, cz+height), color="black", linewidth=1)
    ax3.set_xlabel("X"); ax3.set_ylabel("Y"); ax3.set_zlabel("Z")
    span = max(2*radius, height, 4)
    ax3.set_xlim(cx-span/2, cx+span/2)
    ax3.set_ylim(cy-span/2, cy+span/2)
    ax3.set_zlim(cz, cz+span)
    ax3.set_box_aspect((1, 1, 1))

    if len(xs) == 0:
        status = "Бічна поверхня не перетинається площиною в межах висоти"
    elif np.all(valid):
        status = "Повний замкнений переріз бічної поверхні"
    else:
        status = "Переріз частково виходить за межі циліндра"
    fig.suptitle(f"{status}  |  a={a:.2f}, b={b:.2f}, d={d:.2f}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

interact(
    trainer,
    cx=FloatSlider(min=0, max=10, step=1, value=5, description="Циліндр X"),
    cy=FloatSlider(min=0, max=10, step=1, value=5, description="Циліндр Y"),
    cz=FloatSlider(min=0, max=5, step=1, value=0, description="Циліндр Z"),
    radius=FloatSlider(min=1, max=6, step=.5, value=3, description="Радіус R"),
    height=FloatSlider(min=2, max=10, step=.5, value=6, description="Висота H"),
    a=FloatSlider(min=-2, max=2, step=.1, value=0, description="Нахил a"),
    b=FloatSlider(min=-2, max=2, step=.1, value=0, description="Нахил b"),
    d=FloatSlider(min=-3, max=10, step=.25, value=3, description="Рівень d"),
    show_links=Checkbox(value=True, description="Проекційні орієнтири")
)

interactive(children=(FloatSlider(value=5.0, description='Циліндр X', max=10.0, step=1.0), FloatSlider(value=5…

<function __main__.trainer(cx, cy, cz, radius, height, a, b, d, show_links)>